# Outliers: IQR, média, desvio padrão e Z-score

Este material apresenta duas formas comuns de identificar **outliers** (valores atípicos):

1. **IQR (Intervalo Interquartil)** — baseado em quartis.
2. **Z-score** — baseado na média e no desvio padrão.

> Identificar um outlier não significa automaticamente que ele deve ser removido. A decisão depende do contexto dos dados.

## O que são outliers?

Um **outlier** é uma observação que está muito distante do comportamento predominante dos demais valores.

Exemplo:

```text
10  11  12  11  13  12  10  11  80
```

O valor `80` é um possível outlier.

Outliers podem surgir por:

- erro de medição;
- erro de digitação;
- problema na coleta;
- comportamento realmente raro;
- característica legítima da população.

## IQR — Intervalo Interquartil

O **IQR (Interquartile Range)** representa a distância entre o terceiro e o primeiro quartil:

$$IQR = Q_3 - Q_1$$

- **Q1:** percentil 25.
- **Q2:** percentil 50, ou mediana.
- **Q3:** percentil 75.

Assim, o IQR representa a região dos **50% centrais dos dados**.

### Regra de 1,5 × IQR

$$Limite_{inferior} = Q_1 - 1.5 \times IQR$$

$$Limite_{superior} = Q_3 + 1.5 \times IQR$$

Um valor é um possível outlier quando:

$$x < Limite_{inferior}$$

ou

$$x > Limite_{superior}$$

## Exemplo manual com IQR

Considere:

```text
10, 11, 12, 12, 13, 14, 15, 16, 50
```

Para este exemplo:

```text
Q1 = 12
Q3 = 15
```

Logo:

```text
IQR = 15 - 12
IQR = 3
```

Limites:

```text
Inferior = 12 - 1.5 × 3 = 7.5
Superior = 15 + 1.5 × 3 = 19.5
```

Como `50 > 19.5`, o valor `50` é um possível outlier pelo método do IQR.

## Média

A **média aritmética** é obtida somando todos os valores e dividindo pela quantidade de observações:

$$\bar{x} = \frac{x_1 + x_2 + \dots + x_n}{n}$$

### Atenção aos outliers

A média é **sensível a valores extremos**. Um único valor muito grande ou muito pequeno pode alterar bastante a média.

## Desvio padrão

O **desvio padrão** mede o quanto os valores estão espalhados em relação à média.

- Desvio padrão pequeno → valores mais próximos da média.
- Desvio padrão grande → valores mais espalhados.

Para uma população:

$$\sigma = \sqrt{\frac{1}{N}\sum_{i=1}^{N}(x_i-\mu)^2}$$

## 7. Z-score

O **Z-score** indica quantos desvios padrão uma observação está distante da média:

$$z = \frac{x - \mu}{\sigma}$$

Onde:

- `x` = valor observado;
- `μ` = média;
- `σ` = desvio padrão.

Interpretação:

```text
z = 0   → exatamente na média
z = 1   → 1 desvio padrão acima da média
z = -2  → 2 desvios padrão abaixo da média
```

Quanto maior o valor absoluto de `z`, mais distante a observação está da média.

## Z-score para identificar outliers

Uma regra prática bastante utilizada é:

$$|z| > 3$$

Ou seja:

```text
z < -3
ou
z > 3
```

A regra `|z| > 3` não é universal. A interpretação clássica do Z-score é mais adequada quando os dados têm comportamento aproximadamente normal.

Além disso, como o Z-score utiliza **média e desvio padrão**, essas medidas podem ser influenciadas por valores extremos.

## IQR x Z-score

| Característica | IQR | Z-score |
|---|---|---|
| Base | Quartis | Média e desvio padrão |
| Fórmula | `Q3 - Q1` | `(x - média) / desvio padrão` |
| Regra comum | `1,5 × IQR` | `\|z\| > 3` |
| Sensibilidade a outliers | Menor | Maior |
| Depende de normalidade | Não | A interpretação clássica depende mais dela |
| Usa quartis | Sim | Não |

### Diferença principal

O **IQR** utiliza os quartis e os 50% centrais dos dados, sendo relativamente robusto a valores extremos.

O **Z-score** mede a distância em desvios padrão em relação à média. Como média e desvio padrão podem ser alterados por valores extremos, o próprio critério pode ser afetado por outliers.

## Exemplo completo com Pandas

Agora vamos aplicar os dois métodos à mesma coluna.

In [ ]:
import pandas as pd

data_frame = pd.DataFrame({
    "nome": ["Ana", "Bruno", "Carlos", "Daniela", "Eduardo", "Fernanda", "Gustavo", "Helena"],
    "nota": [8.5, 7.0, 9.2, 6.5, 8.0, 9.0, 7.4, 20.0]
})

data_frame

### Método 1 — IQR

In [ ]:
q1 = data_frame["nota"].quantile(0.25)
q3 = data_frame["nota"].quantile(0.75)
iqr = q3 - q1

limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr

data_frame["outlier_iqr"] = (
    (data_frame["nota"] < limite_inferior) |
    (data_frame["nota"] > limite_superior)
)

print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("Limite inferior:", limite_inferior)
print("Limite superior:", limite_superior)

data_frame

### Método 2 — Z-score

In [ ]:
media = data_frame["nota"].mean()
desvio_padrao = data_frame["nota"].std(ddof=0)

data_frame["z_score"] = (data_frame["nota"] - media) / desvio_padrao
data_frame["outlier_z"] = ((data_frame["z_score"] > 3) | (data_frame["z_score"] < -3))


print("Média:", media)
print("Desvio padrão:", desvio_padrao)

data_frame

In [ ]:
print("Outliers pelo IQR:")
print(data_frame[data_frame["outlier_iqr"]])

print("\nOutliers pelo Z-score:")
print(data_frame[data_frame["outlier_z"]])

## Cuidados ao tratar outliers

Detectar um outlier **não significa que ele deve ser removido**.

Antes de tomar uma decisão, pergunte:

1. O valor é realmente um erro?
2. O valor representa uma situação possível?
3. Existe uma explicação para ele?
4. O outlier pertence ao mesmo grupo dos demais dados?
5. A análise que será realizada é sensível a esse valor?

Dependendo do contexto, podemos:

- manter o valor;
- corrigir um erro de coleta;
- investigar sua origem;
- analisar os dados com e sem o outlier;
- utilizar medidas mais robustas;
- transformar os dados.

A identificação estatística é apenas uma **etapa da análise**.

## Conclusão

- **IQR:** utiliza quartis e é relativamente robusto contra valores extremos.
- **Z-score:** mede a distância em desvios padrão em relação à média.
- Ambos são métodos para **identificar possíveis outliers**, e não regras automáticas para excluir dados.